In [3]:
from phd import lexicon, cupt_parser, lcss_lex, utils, oa_indices, lambdacss, scripts, liai
import os

PARSEME_PATH = os.path.join('..', 'data', 'parseme')
PARSEME_VERSION = '1.2'
LANG = 'DE'

TRAIN_CORPUS_PATH = os.path.join(PARSEME_PATH, PARSEME_VERSION, LANG, 'train.gold.dcupt')
DEV_CORPUS_PATH = os.path.join(PARSEME_PATH, PARSEME_VERSION, LANG, 'dev.gold.dcupt')
TRAINDEV_CORPUS_PATH = os.path.join(PARSEME_PATH, PARSEME_VERSION, LANG, 'traindev.gold.dcupt')
TEST_CORPUS_PATH = os.path.join(PARSEME_PATH, PARSEME_VERSION, LANG, 'test.gold.dcupt')

In [4]:
dev_df = cupt_parser.setup_data_noTT(
	DEV_CORPUS_PATH
)
dev_mwes = cupt_parser.get_mwes(dev_df)
dev_inline_mwes = cupt_parser.inline_mwes(dev_mwes)

train_df = cupt_parser.setup_data_noTT(
	TRAIN_CORPUS_PATH
)
train_mwes = cupt_parser.get_mwes(train_df)
train_inline_mwes = cupt_parser.inline_mwes(train_mwes)

traindev_df = cupt_parser.setup_data_noTT(
	TRAINDEV_CORPUS_PATH
)
traindev_mwes = cupt_parser.get_mwes(traindev_df)
traindev_inline_mwes = cupt_parser.inline_mwes(traindev_mwes)

test_df = cupt_parser.setup_data_noTT(
	TEST_CORPUS_PATH
)
test_mwes = cupt_parser.get_mwes(test_df)
test_inline_mwes = cupt_parser.inline_mwes(test_mwes)

In [5]:
print('Number of MWEs in DEV:', len(dev_inline_mwes[0]))
print('Number of MWEs in TRAIN:', len(train_inline_mwes[0]))
print('Number of MWEs in TRAINDEV:', len(traindev_inline_mwes[0]))
print('Number of MWEs in TEST:', len(test_inline_mwes[0]))

Number of MWEs in DEV: 267
Number of MWEs in TRAIN: 2950
Number of MWEs in TRAINDEV: 3217
Number of MWEs in TEST: 824


In [9]:
dev_mwe_types = set(dev_inline_mwes[0])
train_mwe_types = set(train_inline_mwes[0])
traindev_mwe_types = set(traindev_inline_mwes[0])
test_mwe_types = set(test_inline_mwes[0])

print('Number of MWE types in DEV:', len(dev_mwe_types))
print('Number of MWE types in TRAIN:', len(train_mwe_types))
print('Number of MWE types in TRAINDEV:', len(traindev_mwe_types))
print('Number of MWE types in TEST:', len(test_mwe_types))

Number of MWE types in DEV: 232
Number of MWE types in TRAIN: 1590
Number of MWE types in TRAINDEV: 1689
Number of MWE types in TEST: 585


In [18]:
print('Number of MWE type in DEV and in TRAIN:', len(dev_mwe_types & train_mwe_types))
print('Number of MWE type in DEV and in TEST:', len(dev_mwe_types & test_mwe_types))
print('Number of MWE type in TRAIN and in TEST:', len(train_mwe_types & test_mwe_types))
print('Number of MWE type in TRAINDEV and in TEST:', len(traindev_mwe_types & test_mwe_types))
print()
print('Number of MWE type in DEV and not in TRAIN:', len(dev_mwe_types - train_mwe_types))
print('Number of MWE type in DEV and not in TEST:', len(dev_mwe_types - test_mwe_types))
print('Number of MWE type in TRAIN and not in DEV:', len(train_mwe_types - dev_mwe_types))
print('Number of MWE type in TRAIN and not in TEST:', len(train_mwe_types - test_mwe_types))
print('Number of MWE type in TRAINDEV and not in TEST:', len(traindev_mwe_types - test_mwe_types))
print('Number of MWE type in TEST and not in TRAIN:', len(test_mwe_types - train_mwe_types))
print('Number of MWE type in TEST and not in DEV:', len(test_mwe_types - dev_mwe_types))
print('Number of MWE type in TEST and not in TRAINDEV:', len(test_mwe_types - traindev_mwe_types))
print('Number of MWE type in TEST and not in TRAINDEV:', len(test_mwe_types - (dev_mwe_types | train_mwe_types)))

Number of MWE type in DEV and in TRAIN: 133
Number of MWE type in DEV and in TEST: 85
Number of MWE type in TRAIN and in TEST: 289
Number of MWE type in TRAINDEV and in TEST: 302

Number of MWE type in DEV and not in TRAIN: 99
Number of MWE type in DEV and not in TEST: 147
Number of MWE type in TRAIN and not in DEV: 1457
Number of MWE type in TRAIN and not in TEST: 1301
Number of MWE type in TRAINDEV and not in TEST: 1387
Number of MWE type in TEST and not in TRAIN: 296
Number of MWE type in TEST and not in DEV: 500
Number of MWE type in TEST and not in TRAINDEV: 283
Number of MWE type in TEST and not in TRAINDEV: 283


In [21]:
from collections import Counter
cats = Counter()
for (_, mwe_id), group in dev_mwes.groupby(level=[0, 2]):
	for val in group['parseme:mwe']:
		import re
		m = re.search(rf'(?:^|;){re.escape(str(mwe_id))}:([^;]+)', val)
		if m:
			cats[m.group(1)] += 1
			break
print(cats)

Counter({'VPC.full': 122, 'VID': 95, 'LVC.full': 26, 'IRV': 14, 'VPC.semi': 8, 'LVC.cause': 2})


In [37]:
dev_mwe_types2 = dev_mwes.groupby(level=[0, 2]).apply(
		lambda x : (utils.fmset(list(x['lemma'])),  x['parseme:mwe'].iloc[0].split(':')[-1])
	)

train_mwe_types2 = train_mwes.groupby(level=[0, 2]).apply(
		lambda x : (utils.fmset(list(x['lemma'])),  x['parseme:mwe'].iloc[0].split(':')[-1])
	)

In [42]:
len(set(dev_mwe_types2.to_list()) - set(train_mwe_types2.to_list()))

102